# Train the last layer

## Imports 

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "C:\\Users\\sysadmin\\Documents\\stage\\Edge AI\\KWS\\52-Hertz-KWS\\src")))

In [14]:
import tensorflow as tf
import numpy as np
import os, glob
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from preprocessing import AudioPipeline
from preprocessing.dataset_pipeline import load_and_preprocess, make_dataset
from preprocessing.transfer_model_resnetv2s2 import build_transfer_model2
from sklearn.metrics import classification_report, confusion_matrix

## Config

In [15]:
DATA_DIR    = "C:\\Users\\sysadmin\\Documents\\stage\\Edge AI\\KWS\\52-Hertz-KWS\\data\\raw"
CLASSES     = ["talky", "silence", "unknown"]
BATCH_SIZE  = 16      
EPOCHS   = 20       # backbone gelé
LR       = 1e-3
KERAS_PATH  = "C:\\Users\\sysadmin\\Documents\\stage\\Edge AI\\KWS\\52-Hertz-KWS\\src\\models\\miniresnetv2_s2_64x50_tl.keras"

pipeline = AudioPipeline("C:\\Users\\sysadmin\\Documents\\stage\\Edge AI\\KWS\\52-Hertz-KWS\\configuration\\config.yaml")

## Chargement des chemins + labels 

In [16]:
all_paths, all_labels = [], []

for label_idx, class_name in enumerate(CLASSES):
    wavs = glob.glob(os.path.join(DATA_DIR, class_name, "*.wav"))
    all_paths  += wavs
    all_labels += [label_idx] * len(wavs)
    print(f"  {class_name:10s} → {len(wavs)} fichiers")

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
print(f"\nTotal : {len(all_paths)} samples\n")

  talky      → 295 fichiers
  silence    → 135 fichiers
  unknown    → 326 fichiers

Total : 756 samples



## Split 

In [17]:
# train 70% / val 15% / test 15%
X_train, X_temp, y_train, y_temp = train_test_split(
    all_paths, all_labels,
    test_size=0.30,
    stratify=all_labels,    
    random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}")


Train : 529 | Val : 113 | Test : 114


## Class weights 

In [18]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print(f"\nClass weights : {class_weight_dict}")


Class weights : {0: 0.8559870550161812, 1: 1.856140350877193, 2: 0.7733918128654971}


## tf.data pipeline 

In [19]:
train_ds = make_dataset(X_train, y_train, BATCH_SIZE, augment=True)
val_ds   = make_dataset(X_val,   y_val, BATCH_SIZE,  augment=False)
test_ds  = make_dataset(X_test,  y_test, BATCH_SIZE, augment=False)

## model 

In [13]:
from keras.models import load_model
base_model = load_model(
    KERAS_PATH,
    compile=False  # on recompile nous-mêmes après
)

base_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_2 (InputLayer)          │ (None, 64, 50, 1)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 70, 56, 1)         │               0 │ input_2[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 32, 25, 64)        │           3,200 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 34, 27, 64)        │               0 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 16, 13, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_preact_bn        │ (None, 16, 13, 64)        │             256 │ pool1_pool[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_preact_relu      │ (None, 16, 13, 64)        │               0 │ conv2_block1_preact_bn[0]… │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 16, 13, 64)        │           4,096 │ conv2_block1_preact_relu[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 16, 13, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 16, 13, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_pad            │ (None, 18, 15, 64)        │               0 │ conv2_block1_1_relu[0][0]  │
│ (ZeroPadding2D)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 16, 13, 64)        │           4,160 │ conv2_block1_preact_relu[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 16, 13, 64)        │          36,864 │ conv2_block1_2_pad[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_out (Add)        │ (None, 16, 13, 64)        │               0 │ conv2_block1_0_conv[0][0], │
│                               │                           │                 │ conv2_block1_2_conv[0][0]  │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 440,906 (1.68 MB)

 Trainable params: 20,490 (80.04 KB)

 Non-trainable params: 420,416 (1.60 MB)

In [21]:

model = build_transfer_model2(KERAS_PATH, num_classes=3)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## Callbacks 

In [22]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model_resnet_v2s2.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,                  # généreux : courbes bruitées avec peu de data
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(log_dir="./logs")
]


## Training

In [23]:
print("\n Feature extraction (backbone gelé)")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict   # compense le déséquilibre
)


 Feature extraction (backbone gelé)
Epoch 1/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3289 - loss: 1.1055
Epoch 1: val_accuracy improved from None to 0.66372, saving model to best_model_resnet_v2s2.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 21s 86ms/step - accuracy: 0.3970 - loss: 1.0827 - val_accuracy: 0.6637 - val_loss: 1.0476 - learning_rate: 0.0010
Epoch 2/20
33/34 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.5264 - loss: 1.0441
Epoch 2: val_accuracy improved from 0.66372 to 0.77876, saving model to best_model_resnet_v2s2.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 7s 64ms/step - accuracy: 0.5520 - loss: 1.0424 - val_accuracy: 0.7788 - val_loss: 1.0146 - learning_rate: 0.0010
Epoch 3/20
33/34 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5167 - loss: 1.0011
Epoch 3: val_accuracy did not improve from 0.77876
34/34 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - accuracy: 0.5388 - loss: 1.0012 - val_accuracy: 0.7080 - val_loss: 1.0069 - learning_rate: 0.0010
Epoch 4/20
32/34 ━━━━━━━━━━━━━━━━━━━━

## Best model 

In [25]:
model = tf.keras.models.load_model("best_model_resnet_v2s2.keras")  
loss, acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy : {acc:.3f}")

Test accuracy : 0.930


## Confusion matrix

In [26]:

y_pred = np.argmax(model.predict(test_ds), axis=1)
y_true = np.concatenate([y for _, y in test_ds], axis=0)
print(classification_report(y_true, y_pred, target_names=["talky","silence","unknown"]))

8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 108ms/step
              precision    recall  f1-score   support

       talky       0.93      0.93      0.93        45
     silence       0.80      1.00      0.89        20
     unknown       1.00      0.90      0.95        49

    accuracy                           0.93       114
   macro avg       0.91      0.94      0.92       114
weighted avg       0.94      0.93      0.93       114

